<a href="https://colab.research.google.com/github/TomazDrumond/Tom_Fly/blob/main/NB4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TomazDrumond/Tom_Fly/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import os, sys
import pandas as pd


IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")  # <-- the NAME of the secret you created in Colab's key panel
else:
    hf_token = os.environ.get("HF_TOKEN")

from huggingface_hub import HfApi
api = HfApi(token=hf_token)
files = api.list_repo_files("FlyRank/internship-warehouse", repo_type="dataset")
for f in sorted(files):
    print(f)

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/mont

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

1. **One row means:** one (client, page) combination, summarized over one calendar month
   (2026-03) — i.e., "how did this page perform, for this client, during March 2026."
2. **Table(s):** `fact_content_daily_performance` (month=2026-03 partition, aggregated from
   daily to monthly grain), joined with `dim_content` (page attributes) and `dim_clients`
   (for the availability filter — see Section 3).
3. **Time window:** March 2026, a mid-panel month — chosen deliberately over `_sample`/June
   2026, which is the sealed test month and the natural outcome window for any
   past→future label. Per `dim_clients.gsc_data_start`, history depth differs per client, so
   this global calendar window will be checked against per-client availability in Section 3,
   not assumed uniform.

In [ ]:
rel = "hf://datasets/FlyRank/internship-warehouse"

df_content = pd.read_parquet(f"{rel}/dim_content.parquet", storage_options={"token": hf_token})
print("--- dim_content ---")
print(df_content.shape)
print(df_content.dtypes)
print(df_content.head(3))
print()

df_clients = pd.read_parquet(f"{rel}/dim_clients.parquet", storage_options={"token": hf_token})
print("--- dim_clients ---")
print(df_clients.shape)
print(df_clients.dtypes)
print(df_clients.head(3))
print()

df_daily = pd.read_parquet(
    f"{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet",
    storage_options={"token": hf_token}
)
print("--- fact_content_daily_performance (month=2026-03) ---")
print(df_daily.shape)
print(df_daily.dtypes)
print(df_daily.head(3))
print()

df_query = pd.read_parquet(f"{rel}/fact_content_query_90d.parquet", storage_options={"token": hf_token})
print("--- fact_content_query_90d ---")
print(df_query.shape)
print(df_query.dtypes)
print(df_query.head(3))

--- dim_content ---
(519606, 26)
client_hash_id                 object
content_hash_id                object
keyword_hash_id                object
url_hash_id                    object
keyword_char_count              int64
keyword_token_count             int64
url_char_count                  int64
content_created_date           object
content_updated_date           object
content_type                   object
search_volume                 float64
competition                   float64
competition_level              object
cpc                           float64
main_intent                    object
backlinks                     float64
category_count                  int64
keyword_created_date           object
provider_used                  object
model_used                     object
char_count                    float64
word_count                    float64
last_optimized_date            object
optimization_eligible_date     object
is_published                     bool
is_deleted       

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Feature** (knowable at the decision moment):
- `gsc_impressions`, `gsc_clicks`, `gsc_avg_position` (aggregated to month) — logged GSC data
- `word_count`, `content_updated_date` → `days_since_update` — static/CMS facts
- `content_type` — static page attribute

**Label / proxy** (what we'd predict — kept separate from features):
- `ctr_month = gsc_clicks / gsc_impressions`, thresholded against the page population's
  median, among pages with nonzero impressions — the "needs refresh" proxy.

**Context** (used to build the slice, not fed to a model):
- `client_hash_id`, `content_hash_id` — pseudonymous IDs, grouping/joining only
- `dim_clients.gsc_data_start`, `has_gsc_access` — used to filter, not as features

**Excluded, with why:**
- `fact_content_query_90d` — window (Apr–Jun 2026) overlaps the sealed test month; using it
  now risks leaking future information into features meant to generalize to March.
- `ga4_*` columns — this month's partition shows `client_has_ga4 = False` for many rows and
  `ga4_data_available` is a mixed/None column; including GA4 features without checking each
  client's `ga4_data_start` would silently zero-fill non-adopters as "no engagement" rather
  than "no data" (the exact gotcha the skill warns about) — deferred to a later notebook.

In [ ]:
# Confirm every field named in the buckets above actually exists, spelled correctly

feature_fields = ["gsc_impressions", "gsc_clicks", "gsc_avg_position",
                   "word_count", "content_updated_date", "content_type"]
label_fields = ["gsc_clicks", "gsc_impressions"]  # ctr_month is derived from these
context_fields = ["client_hash_id", "content_hash_id", "gsc_data_start", "has_gsc_access"]

daily_cols = set(df_daily.columns)
content_cols = set(df_content.columns)
clients_cols = set(df_clients.columns)

all_named = set(feature_fields + label_fields + context_fields)
all_available = daily_cols | content_cols | clients_cols

missing = all_named - all_available
print("Fields named above but NOT found in any loaded table:", missing or "none — all confirmed")

print("\nSource check per field:")
for f in sorted(all_named):
    where = [name for name, cols in
             [("fact_content_daily_performance", daily_cols),
              ("dim_content", content_cols),
              ("dim_clients", clients_cols)]
             if f in cols]
    print(f"  {f:25s} -> {where if where else 'NOT FOUND'}")

Fields named above but NOT found in any loaded table: none — all confirmed

Source check per field:
  client_hash_id            -> ['fact_content_daily_performance', 'dim_content', 'dim_clients']
  content_hash_id           -> ['fact_content_daily_performance', 'dim_content']
  content_type              -> ['dim_content']
  content_updated_date      -> ['dim_content']
  gsc_avg_position          -> ['fact_content_daily_performance']
  gsc_clicks                -> ['fact_content_daily_performance']
  gsc_data_start            -> ['dim_clients']
  gsc_impressions           -> ['fact_content_daily_performance']
  has_gsc_access            -> ['dim_clients']
  word_count                -> ['dim_content']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
import duckdb

con = duckdb.connect()
con.register("daily", df_daily)
con.register("clients", df_clients)
con.register("content", df_content)

# --- Query 1: GRAIN CHECK ---
# Claim: one row = one (client, content, report_date)
grain_check = con.sql("""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS n
    FROM daily
    GROUP BY client_hash_id, content_hash_id, report_date
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print("Grain violations (should be empty):")
print(grain_check)
print()

# --- Query 2: ROW COUNT & DATE SPAN ---
span_check = con.sql("""
    SELECT COUNT(*) AS n_rows,
           COUNT(DISTINCT content_hash_id) AS n_pages,
           COUNT(DISTINCT client_hash_id) AS n_clients,
           MIN(report_date) AS first_day,
           MAX(report_date) AS last_day
    FROM daily
""").df()
print("Row count & date span:")
print(span_check)
print()

# --- Query 3: AVAILABILITY, filtered with IS TRUE ---
availability_check = con.sql("""
    SELECT COUNT(*) AS total_rows,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows
    FROM daily
""").df()
print("Availability check:")
print(availability_check)
share = availability_check["gsc_available_rows"][0] / availability_check["total_rows"][0]
print("Share with GSC data available:", round(share, 3))
print()

# --- Query 4: PER-CLIENT HISTORY DEPTH (the panel warning from the skill) ---
# Claim: history depth differs per client, so a flat March-2026 window may include
# clients who don't actually have GSC data yet that month.
per_client_check = con.sql("""
    SELECT c.client_hash_id, c.gsc_data_start,
           COUNT(d.report_date) AS march_rows_for_client
    FROM clients c
    LEFT JOIN daily d ON c.client_hash_id = d.client_hash_id
    WHERE c.gsc_data_start IS NULL OR c.gsc_data_start > '2026-03-01'
    GROUP BY c.client_hash_id, c.gsc_data_start
    ORDER BY march_rows_for_client DESC
    LIMIT 10
""").df()
print("Clients with gsc_data_start after March 2026 (or missing) — check if they still show March rows:")
print(per_client_check)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain violations (should be empty):
Empty DataFrame
Columns: [client_hash_id, content_hash_id, report_date, n]
Index: []



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Row count & date span:
    n_rows  n_pages  n_clients  first_day   last_day
0  9841378   331437         55 2026-03-01 2026-03-31

Availability check:
   total_rows  gsc_available_rows
0     9841378             3611061
Share with GSC data available: 0.367

Clients with gsc_data_start after March 2026 (or missing) — check if they still show March rows:
            client_hash_id gsc_data_start  march_rows_for_client
0  client_19b89ee4fe3db6da            NaT                 213001
1  client_b77d0d5f08f05e64     2026-03-12                  19440
2  client_86ebc2f12c01f586     2026-03-11                   8823
3  client_80ee5b7bd5f4eb89            NaT                   6076
4  client_810019792c9b8efc     2026-03-25                   1812
5  client_e00b29e582949543     2026-03-27                   1216
6  client_f6f0cdf26d03d7bd     2026-03-19                    520
7  client_7de9989c909e91a5     2026-05-18                      0
8  client_46703b915c4762e0            NaT                     

## 2b. Five-feature frame + leakage check

*Build the actual monthly page-level feature frame, then spring the deliberate leak.*

In [ ]:
# Build the actual monthly page-level feature frame from March 2026
feature_frame = con.sql("""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS impressions_month,
           SUM(gsc_clicks) AS clicks_month,
           AVG(gsc_avg_position) AS avg_position_month
    FROM daily
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

feature_frame = feature_frame.merge(
    df_content[["client_hash_id", "content_hash_id", "content_updated_date", "word_count"]],
    on=["client_hash_id", "content_hash_id"], how="left"
)

feature_frame["content_updated_date"] = pd.to_datetime(feature_frame["content_updated_date"])
snapshot_date = pd.Timestamp("2026-03-31")
feature_frame["days_since_update"] = (snapshot_date - feature_frame["content_updated_date"]).dt.days

feature_cols = ["impressions_month", "clicks_month", "avg_position_month",
                 "days_since_update", "word_count"]
print(feature_frame.shape)
feature_frame[feature_cols].describe()


Five features, each with an "available when?" line:
- `impressions_month` — knowable at month-end: it's the logged GSC total for the month.
- `clicks_month` — knowable at month-end: logged as it happens through the month.
- `avg_position_month` — knowable at month-end: GSC reports position daily.
- `days_since_update` — knowable before the snapshot: content_updated_date is a CMS fact.
- `word_count` — knowable at publish time: a static property of the page.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split

visible = feature_frame[feature_frame["impressions_month"] > 0].copy()
visible["ctr_month"] = visible["clicks_month"] / visible["impressions_month"]
median_ctr = visible["ctr_month"].median()
visible["needs_refresh_label"] = (visible["ctr_month"] < median_ctr).astype(int)

y = visible["needs_refresh_label"]

# Honest model — no clicks_month, since it defines the label
X_honest = visible[["impressions_month", "avg_position_month", "days_since_update", "word_count"]].fillna(0)
Xtr, Xte, ytr, yte = train_test_split(X_honest, y, test_size=0.3, random_state=42)
tree = DecisionTreeClassifier(max_depth=3, random_state=42).fit(Xtr, ytr)
print("Honest score:", tree.score(Xte, yte))

# Deliberate leak — clicks_month is literally used to compute the label
X_leaky = visible[["impressions_month", "avg_position_month", "days_since_update",
                    "word_count", "clicks_month"]].fillna(0)
Xtr_l, Xte_l, ytr_l, yte_l = train_test_split(X_leaky, y, test_size=0.3, random_state=42)
tree_leaky = DecisionTreeClassifier(max_depth=3, random_state=42).fit(Xtr_l, ytr_l)
print("Leaky score (clicks_month included):", tree_leaky.score(Xte_l, yte_l))


Adding `clicks_month` pushed the score toward-perfect, because `clicks_month` is a direct
component of `ctr_month`, which the label is thresholded on — the model isn't learning a
pattern, it's reconstructing the label's own arithmetic. Deleting `clicks_month` and keeping
only the four remaining features restores the honest, lower score above — that's the number
to report and build on going forward.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This slice has three real limits, confirmed by the queries above, not assumed:

1. **Only 36.7% of the 9.8M rows have `gsc_data_available IS TRUE`.** The daily table isn't
   a table of "observed events" — it looks like a full scaffold of (client, page, day)
   combinations for the month, most of which have no real GSC data behind them. Any query
   that doesn't filter on `gsc_data_available IS TRUE` will silently treat non-observations
   as zero-activity observations, which is a different, wrong claim.

2. **`gsc_data_start` doesn't cleanly predict which rows are real.** Some clients with no
   recorded `gsc_data_start` (NaT) still produced tens of thousands of March rows
   (e.g., client `19b89ee4...`: 213,001 rows) — these are very likely present-but-unavailable
   rows, not genuine history. Meanwhile a client starting mid-month (`2026-03-12`) still
   shows a full month's worth of row *slots* (19,440), meaning the row count alone never
   tells you about real coverage — only the `IS TRUE` flag does.

3. **History depth is genuinely uneven across the panel, and it's not a small effect.**
   8 of 55 clients (14.5%) have 0% GSC availability in March — meaning they contribute rows
   to this slice but zero usable signal. Zero clients have full (100%) availability. The
   median client sits at only 8.0% availability, while the mean (24.6%) is pulled up by a
   right tail reaching 91.6% at the top — so "availability" is heavily skewed, not evenly
   spread. Any feature or label built from this slice implicitly represents a small,
   high-availability subset of the 55-client panel, not the full panel — that should be
   stated explicitly in any downstream model card, not left implicit.

In [ ]:
# How many of the 55 clients actually have usable GSC data this month vs just "present" rows?
client_summary = con.sql("""
    SELECT client_hash_id,
           COUNT(*) AS total_rows,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS available_rows
    FROM daily
    GROUP BY client_hash_id
""").df()

client_summary["pct_available"] = client_summary["available_rows"] / client_summary["total_rows"]

print("Clients with 0% GSC availability in March:",
      (client_summary["pct_available"] == 0).sum(), "/", len(client_summary))
print("Clients with 100% GSC availability in March:",
      (client_summary["pct_available"] == 1).sum(), "/", len(client_summary))
print("\nDistribution of per-client availability:")
print(client_summary["pct_available"].describe())


Clients with 0% GSC availability in March: 8 / 55
Clients with 100% GSC availability in March: 0 / 55

Distribution of per-client availability:
count    55.000000
mean      0.245580
std       0.299994
min       0.000000
25%       0.016287
50%       0.080149
75%       0.455436
max       0.916412
Name: pct_available, dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.